# **Self-Attention and Positional Encoding**


## Objectives

- Understand the basics of tokenization and how textual data is prepared for neural network models.
- Learn the concept of one-hot encoding and its application in representing textual data for machine learning models.
- Explore the self-attention mechanism, a key component of Transformer models, which allows the model to dynamically focus on different parts of the input sequence for making predictions.
- Implement a basic self-attention mechanism and integrate it into a neural network model.
- Grasp the importance of positional encoding in Transformer models, providing the model with the necessary information about the order of words in a sentence.
- Implement positional encoding and observe its effect on model performance and understanding of sequence order.
- Apply the learned concepts to build a simple translation model or text processing task, demonstrating the practical application of self-attention and positional encoding.
- Develop an intuition for the workings of modern NLP models, particularly the Transformer architecture, and understand their advantages over traditional sequence processing models like RNNs and LSTMs.
- Encourage further exploration and experimentation with different model architectures, hyperparameters, and applications in natural language processing.


### **Self-Attention**

Self-attention is a technique used in neural networks that allows a model to focus on different parts of the input while producing each part of the output. It is a fundamental component of the Transformer architecture, which is widely applied in natural language processing tasks such as machine translation, text summarization, and sentiment analysis.

The core idea of self-attention is to enable the model to determine how important each input token is when generating an output token. This is achieved by calculating a weighted combination of all input tokens, where the weights reflect the relationships between every pair of tokens.



- [`torch`](https://pytorch.org/): The core library for building and training neural network models in this project, including the implementation of Self-Attention mechanisms and Positional Encodings.
- [`torch.nn`](https://pytorch.org/docs/stable/nn.html), [`torch.nn.functional`](https://pytorch.org/docs/stable/nn.functional.html): These PyTorch submodules are used to define the neural network layers and apply functions such as activations, which are essential in building the model architecture.
- [`Levenshtein`](https://pypi.org/project/python-Levenshtein/): This library is used for calculating the Levenshtein distance, which can be useful for evaluating model performance in tasks like text generation or translation by measuring the difference between the predicted and actual text sequences.
- [`get_tokenizer`](https://pytorch.org/text/stable/data_utils.html), [`build_vocab_from_iterator`](https://pytorch.org/text/stable/vocab.html) from `torchtext`: These functions are crucial for preprocessing text data, including tokenizing text into words or subwords and building a vocabulary from the dataset, which are foundational steps in preparing data for NLP models.


### **Import Libraries**

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.functional as F
from torchtext.vocab import build_vocab_from_iterator
from torchtext.data.utils import get_tokenizer

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


- **Training parameters**: 
  - `learning_rate`: This is the step size at each iteration while moving toward a minimum of the loss function. We've set it to `3e-4`, which is a common starting point for many models.
  - `batch_size`: The number of samples that will be propagated through the network in one forward/backward pass. Here, it's `64`.
  - `max_iters`: The total number of training iterations we plan to run. Set to `5000` to allow the model ample opportunity to learn from the data.
  - `eval_interval` and `eval_iters`: Parameters defining how frequently we evaluate the model's performance on a set number of batches to approximate loss.

- **Architecture parameters**: 
  - `max_vocab_size`: This represents the maximum number of tokens in our vocabulary. It's set to `256`, meaning that we will only consider the most frequent 256 tokens.
  - `vocab_size`: The actual number of tokens in the vocabulary, which may be less than the maximum due to the variable length of tokens in subword tokenization like BPE (Byte Pair Encoding).
  - `block_size`: The length of the input sequence that the model is designed to handle. Here it's `16`.
  - `n_embd`: The size of each embedding vector, set to `32`. Embeddings convert tokens into a continuous space where similar tokens are closer to each other.
  - `num_heads`: The number of heads in the multi-headed self-attention mechanism, `2` in this case, which allows the model to jointly attend to information from different representation subspaces.
  - `n_layer`: The number of layers (or depth) of the network. Here, `2` layers are used.
  - `ff_scale_factor`: A scaling factor for the size of the feed-forward networks, chosen as `4` here.
  - `dropout`: The dropout rate used for regularization to prevent overfitting, set at `0.0`, indicating no dropout in this case.

Finally, you have a `head_size` calculation that is derived from the embedding size and number of heads, ensuring that each head has an equal chunk of the embedding size to work with. We also include an assertion to verify that the `head_size` times `num_heads` equals the `n_embd`.


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 

In [3]:
# training parameters
lr = 3e-4
batch_size = 64
max_iter = 5000
eval_intervals = 200
eval_iters = 100

In [4]:
# Model parameters
max_vocab_size = 256
vocab_size = max_vocab_size
block_size = 16
n_embeds = 32
num_heads = 2
n_layers = 2
ff_scale_factor = 4
dropout = 0.0


In [5]:
head_size = n_embeds // num_heads
assert ( num_heads * head_size) == n_embeds

### **Tokenization**

Small translation using the dictionary key value pairs

In [6]:
dictionary = {
    'le': 'the'
    , 'chat': 'cat'
    , 'est': 'is'
    , 'sous': 'under'
    , 'la': 'the'
    , 'table': 'table'
}

In [7]:
def tokenization(text):
    tokenizer = get_tokenizer("basic_english")
    tokenized_text = tokenizer(text)
    return tokenized_text
    
def translate(text):
    tokenized_text = tokenization(text)
    translate_text = [dictionary[word] for word in tokenized_text]
    
    out = ""
    for text in translate_text:
        out += text + " "
    return out.strip()
    
    

In [8]:
translate("le chat est sous la table")

'the cat is under the table'

### Improvement: Handling Cases Where the 'key' Is Missing in the Dictionary

This section introduces an enhancement to the translation system to manage situations where a word is not found in the dictionary:

- **find_closest_key Function**: This newly introduced function is designed to identify the closest matching key in the dictionary for a given query word. It utilizes the **Levenshtein distance** (a metric that measures the difference between two sequences) to determine the dictionary entry with the smallest distance to the input word, enabling it to suggest a similar alternative when an exact match is unavailable.

- **Enhanced translate Function**: The `translate` function has been modified to incorporate `find_closest_key`. Instead of directly mapping tokens using the dictionary, it first determines the closest matching key for each tokenized word. This improvement makes the translation process more resilient, particularly when dealing with slight spelling mistakes or variations that are not explicitly included in the dictionary.

- **Example Demonstration**: The updated translate function is demonstrated using the input "tables". Even though "tables" is not present in the dictionary, the function successfully identifies "table" as the closest match and uses it for translation, producing "table" in English.

This enhancement demonstrates a basic approach to error handling and fuzzy matching in translation systems, resulting in more adaptable and tolerant translations.

In [9]:
from Levenshtein import distance

In [10]:
def closet_key(query, dictionary):
    closest_dis = float('inf')
    closet_key = None
    
    for key in dictionary.keys():
        #get the distance between query and keys using levenshtein distance
        dist = distance(query,key)
        
        if dist < closest_dis:
            closest_dis = dist
            closet_key = key
    
    return closet_key, closest_dis

In [11]:
def translate_sentence (sentence):
    
    out = ''
    
    for query in tokenization(sentence):
        key,dis = closet_key(query,dictionary)
        out += dictionary[key] + " "
    
    return out.strip()

In [12]:
translate_sentence("cats")

'cat'

## Convert to neural network

Moving from simple translation methods to neural networks, we begin by defining the input and output vocabularies, followed by encoding the tokens:

- **Vocabulary definition**: Two vocabularies are generated from the dictionary—`vocabulary_in` for the source language (French) and `vocabulary_out` for the target language (English). These vocabularies consist of the unique words extracted from the dictionary’s keys and values, respectively, and are sorted to ensure a consistent order.

- **One-hot encoding**: The `encode_one_hot` function is used to transform each word in the vocabulary into a one-hot encoded vector. In this approach, each word is represented as a binary vector where a '1' appears at the index corresponding to that word in the vocabulary, while all other positions are filled with '0's. This results in a distinct, fixed-length vector for every word, making it suitable for neural network input.

- **Encoding demonstration**: The one-hot encoding process is illustrated by applying `encode_one_hot` to the input vocabulary (`vocabulary_in`) and displaying the resulting vectors for each word. The same procedure is then repeated for the output vocabulary (`vocabulary_out`).

This step is essential in machine learning because it converts textual data into a numerical format that neural networks can process, enabling them to learn patterns and generate predictions.

### **Vocabulary Defining**

In [13]:
vocab_in = sorted(list(set(dictionary.keys())))

print(f"vocab length: ({len(vocab_in)}): ({vocab_in})")

vocab length: (6): (['chat', 'est', 'la', 'le', 'sous', 'table'])


In [14]:
vocab_out = sorted(list(set(dictionary.values())))

print(f"vocab out:({len(vocab_out)}): ({vocab_out})")

vocab out:(5): (['cat', 'is', 'table', 'the', 'under'])


In [15]:
def encode_one_hot(vocabulary):
    
    vocabulary_len = len(vocabulary)
    one_hot = dict()
    
    for i, key in enumerate(vocabulary):
        one_hot_vector = torch.zeros(vocabulary_len)
        one_hot_vector[i] = 1
        one_hot[key] = one_hot_vector

        
    return one_hot

In [36]:
input_encode_one= encode_one_hot(vocab_in)
input_encode_one

{'chat': tensor([1., 0., 0., 0., 0., 0.]),
 'est': tensor([0., 1., 0., 0., 0., 0.]),
 'la': tensor([0., 0., 1., 0., 0., 0.]),
 'le': tensor([0., 0., 0., 1., 0., 0.]),
 'sous': tensor([0., 0., 0., 0., 1., 0.]),
 'table': tensor([0., 0., 0., 0., 0., 1.])}

In [53]:
output_encode_one = encode_one_hot(vocab_out)
output_encode_one

{'cat': tensor([1., 0., 0., 0., 0.]),
 'is': tensor([0., 1., 0., 0., 0.]),
 'table': tensor([0., 0., 1., 0., 0.]),
 'the': tensor([0., 0., 0., 1., 0.]),
 'under': tensor([0., 0., 0., 0., 1.])}

### Iterate Over Encoding Vectors

In [18]:
for k,v in enumerate(input_encode_one):
    print(f"key:{k}: value:{v}")

key:0: value:chat
key:1: value:est
key:2: value:la
key:3: value:le
key:4: value:sous
key:5: value:table


In [21]:
for k,v in enumerate(output_encode_one):
    print(f"key:{k}: value:{v}")

key:0: value:cat
key:1: value:is
key:2: value:table
key:3: value:the
key:4: value:under


## **Matrix Multiplication**

This section we focus on create a dictionary using matrix multiplication

### Let's create a 'dictionary' using matrix multiplication

We're now illustrating how to create a representation of our dictionary suitable for neural network operations:

- **Matrix creation**: Using PyTorch's `torch.stack`, convert the one-hot encoded vectors for both input (`K`) and output (`V`) vocabularies into tensors. `K` is constructed from the input vocabulary's one-hot vectors, and `V` from the output vocabulary's vectors. These tensors can be thought of as a look-up table that our model will use to associate input tokens with output tokens.

- **Dictionary as matrices**: This step effectively translates our word-to-word dictionary mapping into a neural network-friendly format. Each row in `K` corresponds to a word in the input language represented as a one-hot vector, and each row in `V` corresponds to the respective translated word in the output language.

- **Query example**: An example shows how to use matrix operations to find a translation. Look up the one-hot vector for the word "sous" from the input vocabulary (`q`). Then demonstrate how to find its corresponding translation by performing matrix multiplication with the transpose of `K` (i.e., `q @ K.T`) to identify the index and then use that index to select the relevant row from `V`. This process mimics the lookup the you would perform in an actual neural network during translation tasks.

This matrix representation is a precursor to understanding how more complex neural network architectures, like those using self-attention, manage token translations.


In [41]:
# key vector
k = torch.stack([input_encode_one[k] for k in dictionary.keys()])

print(f"key: {k}")

key: tensor([[0., 0., 0., 1., 0., 0.],
        [1., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0.],
        [0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1.]])


In [42]:
v = torch.stack([output_encode_one[k] for k in dictionary.values()])

print(f"values: {v}")

values: tensor([[0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 1., 0.],
        [0., 0., 1., 0., 0.]])


In [48]:
# query
q = (input_encode_one['sous']).unsqueeze(0)
print("query: ",q)

query:  tensor([[0., 0., 0., 0., 1., 0.]])


In [49]:
# finding the selected key
selected_key = q @ k.T
print("selected key: ", selected_key)

selected key:  tensor([[0., 0., 0., 1., 0., 0.]])


In [50]:
# finding relevant value
selected_value = q @ k.T @ v
print(f"selected value: {selected_value}")

selected value: tensor([[0., 0., 0., 0., 1.]])


The operation $q \cdot K^T \cdot V$ allows us to build a dictionary-like structure from a set of vectors

This is an example on how to select the value from a query:

$$
q \cdot K^T \cdot V =
\left[\begin{matrix}
  0 & 0 & 0 & 0 & 1 & 0\\\\\\\\\\\\\\\\
\end{matrix}\right]
\cdot
\left[\begin{matrix}
  0 & 1 & 0 & 0 & 0 & 0\\\\
  0 & 0 & 1 & 0 & 0 & 0\\\\
  0 & 0 & 0 & 0 & 1 & 0\\\\
  1 & 0 & 0 & 0 & 0 & 0\\\\
  0 & 0 & 0 & 1 & 0 & 0\\\\
  0 & 0 & 0 & 0 & 0 & 1\\\\
\end{matrix}\right]
\cdot
\left[\begin{matrix}
  0 & 0 & 0 & 1 & 0\\\\
  1 & 0 & 0 & 0 & 0\\\\
  0 & 1 & 0 & 0 & 0\\\\
  0 & 0 & 0 & 0 & 1\\\\
  0 & 0 & 0 & 1 & 0\\\\
  0 & 0 & 1 & 0 & 0\\\\
\end{matrix}\right]
$$


$$
q \cdot K^T \cdot V =
%\hspace{2cm}
\left[\begin{matrix}
  0 & 0 & 0 & 1 & 0 & 0\\\\\\\\\\\\\\\\
\end{matrix}\right]
%\hspace{2.5cm}
\cdot
\left[\begin{matrix}
  0 & 0 & 0 & 1 & 0\\\\
  1 & 0 & 0 & 0 & 0\\\\
  0 & 1 & 0 & 0 & 0\\\\
  0 & 0 & 0 & 0 & 1\\\\
  0 & 0 & 0 & 1 & 0\\\\
  0 & 0 & 1 & 0 & 0\\\\
\end{matrix}\right]
\hspace{4.5cm}
$$


$$
q \cdot K^T \cdot V
=
%\hspace{3.5cm}
\left[\begin{matrix}
0 & 0 & 0 & 0 & 1\\\\\\\\\\\\\\\\
\end{matrix}\right]
%\hspace{3.5cm}
\hspace{9cm}
$$


## **Decode One-Hot Vector**

The decode_one_hot function converts a one-hot encoded vector back into its corresponding token (word). Since a one-hot vector contains a single 1 and the rest 0s, decoding can be done by finding the index of the maximum value (using argmax). This index directly maps to the token in the vocabulary.

Unlike more complex embeddings, cosine similarity is unnecessary here—because one-hot vectors are orthogonal, the position of the 1 uniquely identifies the token.

In [54]:
def decode_one_hot(one_hot,vector):
    
    cosine_max = 0
    best_key = None
    
    for k,v in one_hot.items():
        cosine = vector @ v
        
        if cosine > cosine_max:
            cosine_max = cosine
            best_key = k
    
    return best_key

### **Matrix-based translate function**

The `translate` function now uses matrix operations to perform translation. For each token in the input sentence, it retrieves its one-hot vector, performs matrix multiplication with `K.T` and `V` to compute the corresponding output representation in the target vocabulary space, and then decodes this resulting vector back into the translated word.

In [57]:
def translate_decoder(sentence):
    
    sentence_out = ''
    
    for token in tokenization(sentence):
        
        query  = input_encode_one[token]
        output = query @ k.T @ v
        
        best_key = decode_one_hot(output_encode_one,output)
        sentence_out += best_key + ' '
        
    return sentence_out.strip()  
         

In [58]:
translate_decoder("le chat est sous la table")

'the cat is under the table'

## **Softmax Function**

Softmax function for similarity

Similar tokens tend to have similar vector representations. To capture this relationship, a softmax function is applied to the result of the matrix multiplication between the query vector `q` and the transpose of the key matrix `K`. This softmax operation converts the raw similarity scores into a probability distribution, highlighting the most relevant (most similar) tokens while still assigning smaller weights to the remaining ones.

Our new equation is:
$$
softmax(q \cdot K^T) \cdot V
$$

Let's adjust using by the dimensionality of the query vector, and you'll get:

$$
softmax\left( \frac{q \cdot K^T}{\sqrt{d}} \right) \cdot V
$$


In [ ]:
def translate_softmax (query):
    
    sentence_out = ""
    query_tokenized = tokenization(query)
    
    for token in query_tokenized:
        
        query_value = input_encode_one[token]
        
        output = torch.softmax(query_value @ k.T, dim=0) @ v
        
        decoder_output = decode_one_hot(output_encode_one,output)
        sentence_out += decoder_output + ' '
    
    
    return sentence_out.strip()
        

In [62]:
translate_softmax("le chat est sous la table")

'the cat is under the table'